# Week 7 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-07.md`](../../weeks/week-07.md) · **Given engine:** [`dpll.py`](dpll.py)

This notebook is the **teaching walkthrough** of the week-7 DPLL / Wumpus studio. It
**imports the reference code from [`solution.py`](solution.py)** — it never re-pastes it —
so the agent shown is byte-for-byte the one the provided test grades. The correctness
guarantee is separate: `python3 studios/_verify_solutions.py week-07` runs the unmodified
`test_dpll.py` against `solution.py`.

> Do not distribute. Excluded from students via `studios/.gitignore`.

In [ ]:
# --- bootstrap: put the week folder (for solution/dpll) and the repo root
# (for aicourse) on the path, so this runs from anywhere. ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-07"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect
import dpll
from dpll import (Pv, entails, kb_from_percepts, classify, load_scenarios)
import solution

SC = load_scenarios()["wumpus"]
BASE = SC["base_percepts"]
EXTRA = SC["extra_percept"]
FRONTIER = [tuple(sq) for sq in SC["frontier"]]

## The central identity, in code

`KB ⊨ α  iff  KB ∧ ¬α is unsatisfiable`. It turns a question about *all* models into a
*search* for one — that is what makes a SAT solver a general-purpose reasoning engine.
Modus ponens holds; affirming the consequent does not, and the solver agrees.

In [ ]:
P, Q = 1, 2
mp = [frozenset({-P, Q}), frozenset({P})]         # {P->Q, P}
assert entails(mp, [frozenset({Q})])              # ... entails Q
assert not entails(mp, [frozenset({-Q})])         # ... but not ¬Q
print("ok  modus ponens: {P->Q, P} entails Q")

ac = [frozenset({-P, Q}), frozenset({Q})]         # {P->Q, Q}
assert not entails(ac, [frozenset({P})])          # ... does NOT entail P
print("ok  affirming the consequent {P->Q, Q} does NOT entail P (invalid, correctly)")

## The ASK step — `provably_safe` / `provably_pit`

Each is a thin layer over `entails`: does the KB *prove* there is no pit (resp. a pit) at
`(x,y)`? Note the sign — `¬P(x,y)` is the literal `-Pv(x,y)` wrapped as a one-clause
CNF.

In [ ]:
print(inspect.getsource(solution.provably_safe))
print(inspect.getsource(solution.provably_pit))

kb = kb_from_percepts(BASE)
print("classification on base percepts (1,1) no breeze, (2,1) breeze:")
for (x, y) in FRONTIER:
    print(f"  ({x},{y}): {classify(kb, x, y)}")
# mirrors test_wumpus_base_classification: (1,2) proved safe, rest UNKNOWN
assert solution.provably_safe(kb, 1, 2)
assert not solution.provably_safe(kb, 2, 2)
print("ok  (1,2) is provably safe; (2,2)/(3,1)/(1,3) are UNKNOWN")

### Monotonic reasoning — one extra percept resolves two squares

Adding "no breeze at (1,2)" proves (2,2) and (1,3) SAFE and — because the breeze at (2,1)
must be explained — (3,1) a proved PIT. Adding true facts never invalidates a prior
conclusion (week 12 is where that guarantee is given up).

In [ ]:
kb2 = kb_from_percepts(BASE + EXTRA)
for key, want in SC["expected_after_extra"].items():
    x, y = (int(t) for t in key.split(","))
    got = classify(kb2, x, y)
    assert got == want, f"({key}): expected {want}, got {got}"
    print(f"  ({key}): {got}")
print("ok  monotonic: (2,2) SAFE, (1,3) SAFE, (3,1) PIT")

## The sound agent — moves only to proved-safe squares

`sound_frontier` returns only squares the KB *proves* have no pit; it never treats
UNKNOWN as safe, and returns `[]` when nothing is proved. On the base percepts that is
exactly `{(1,2)}`.

In [ ]:
print(inspect.getsource(solution.sound_frontier))
kb = kb_from_percepts(BASE)
moves = solution.sound_frontier(kb, FRONTIER)
# mirrors test_sound_agent_only_moves_to_proved_safe
for (x, y) in moves:
    assert entails(kb, [frozenset({-Pv(x, y)})])
assert set(moves) == {(1, 2)}
print("sound_frontier:", moves)
print("ok  sound agent moves only to proved-safe squares")

## Guarantee-fail moment — the RECKLESS agent acts unsoundly (on purpose)

This is the point of the week. `reckless_frontier` skips the entailment-for-safety check
and moves anywhere *not proved to be a pit* — i.e. it treats UNKNOWN as safe. The sound
agent structurally *cannot* make that move; watch the unsound one make it. This is exactly
the "probably safe" ≠ "provably safe" error the LLM commits in Task 2.

In [ ]:
print(inspect.getsource(solution.reckless_frontier))
kb = kb_from_percepts(BASE)
reckless = set(solution.reckless_frontier(kb, FRONTIER))
sound = set(solution.sound_frontier(kb, FRONTIER))
unsound = [(x, y) for (x, y) in reckless if not entails(kb, [frozenset({-Pv(x, y)})])]
print("sound   :", sorted(sound))
print("reckless:", sorted(reckless))
print("UNSOUND moves (chosen but NOT provably safe):", sorted(unsound))
# mirrors test_reckless_agent_is_unsound
assert unsound and reckless > sound
print(f"\nGUARANTEE FAIL confirmed: the reckless agent made {len(unsound)} unsound move(s)")
print("onto UNKNOWN squares. Any one of them is a pit it cannot rule out — it dies.")

## Task 2 — the LLM as the reasoner (headless)

Give an LLM the same percepts in natural language and score its "certainly safe" claims
for **soundness** against `sound_frontier` (the ground truth). We run headless with the
`echo` backend so the notebook executes anywhere. **`echo` is a FAKE** — it returns prose,
not a safety verdict; its text must never be scored as a model result. Re-run with
`backend="ollama"` or `backend="manual"` for a real comparison.

Score on **soundness, not accuracy**: saying UNKNOWN when it is UNKNOWN is a *good* result
(epistemic humility); confidently asserting an unprovable claim is the failure that
matters. The DPLL agent has soundness *before* you run anything; the LLM has no soundness
property at all — a categorical difference, not a percentage one.

In [ ]:
from aicourse.llm import LLM
llm = LLM(backend="echo")   # FAKE: not a real safety verdict

PROMPT = '''You are in a 4x4 cave at square (1,1). Rules:
- A square with a breeze is adjacent to at least one pit.
- Adjacent means horizontally or vertically, not diagonally.
Observations so far:
  (1,1): no breeze
  (2,1): breeze
Which squares are certainly safe? For each, state the observation that forces it.'''

reply = llm.complete(PROMPT)
print("echo reply (FAKE):", reply.text)

truth = set(solution.sound_frontier(kb_from_percepts(BASE), FRONTIER))
print("\nground truth (provably safe):", sorted(truth))
print("Scoring recipe: parse the model's 'certainly safe' set, then any square it names")
print("that is NOT in the ground-truth set is an UNSOUND claim — the failure to record.")